# DeltaNet Effective Beta Diagnostics


In [ ]:
from __future__ import annotations

import sys
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
import torch
import torch.nn.functional as F

repo_root = Path.cwd().resolve()
while repo_root != repo_root.parent and not (repo_root / "PFNs").exists():
    repo_root = repo_root.parent
if not (repo_root / "PFNs").exists():
    raise RuntimeError("Could not find repo root containing PFNs/.")

sys.path.insert(0, str(repo_root / "PFNs"))

from fla.layers.delta_net import DeltaNet
from pfns.experiments.model_benchmarks.model_registry import MODEL_FAMILIES
from pfns.experiments.model_benchmarks.models import load_models_for_benchmark
from pfns.model.fla_patches import _apply_deltanet_beta_decay

warnings.filterwarnings(
    "ignore",
    message="ShortConvolution is crucial to the performance.*",
    category=UserWarning,
)
plt.rcParams["figure.dpi"] = 120


## Configuration


In [ ]:
MODEL_NAMES = [
    "Non_Causal_DeltaNet",
    "DeltaNet_Comb_ST_Reference",
    "DeltaNet_Comb_ST_Seq_Len_200-64K_loguniform",
    "Non_Causal_DeltaNet_loguniform_64K",
    "Non_Causal_DeltaNet_with_lr_decay_online_inverse_t0_256",
]
MODEL_LABELS = {
    "Non_Causal_DeltaNet": "NC DeltaNet",
    "Non_Causal_DeltaNet_loguniform_64K": "NC DeltaNet loguniform 64K",
    "DeltaNet_Comb_ST_Reference": "Reference",
    "DeltaNet_Comb_ST_Seq_Len_200-64K_loguniform": "200-64K",
    "Non_Causal_DeltaNet_with_lr_decay_online_inverse_t0_256": "NC online inverse T0=256",
}

LAYER_INDICES = [0, 3, 6, 9, 11]
TRAIN_CONTEXT_LEN = 128_000
N_TEST = 64
BATCH_SIZE = 1
NUM_FEATURES = 10

MAX_CURVE_POINTS = 4096
MAX_MATRIX_POINTS = 1024
POSITION_SAMPLING = "log"  # "log" or "linear"
SMOOTH_BETA = True
SMOOTHING_WINDOW = 101
RETENTION_FLOOR = max(1e-6, float(torch.finfo(torch.float32).eps))
RETENTION_BETA_CEILING = 1.0 - 1e-7
MATRIX_AXIS_SCALES = ["log"] #["log", "linear"]

## Collect Effective Betas


In [ ]:
def model_label(model_name: str) -> str:
    return MODEL_LABELS.get(model_name, model_name)


def find_model_config(model_name: str) -> dict:
    for registry in MODEL_FAMILIES.values():
        if model_name in registry:
            return registry[model_name]
    raise KeyError(f"Could not find model {model_name!r} in MODEL_FAMILIES.")


def load_model(model_name: str):
    loaded_models, loaded_configs = load_models_for_benchmark(
        {model_name: model_configs[model_name]},
        device=str(device),
    )
    model = loaded_models[model_name]
    model.eval()
    return model, loaded_configs[model_name]


def effective_beta(model, hook_output: torch.Tensor) -> torch.Tensor:
    beta = hook_output.detach().float().sigmoid()
    backbone = getattr(model, "transformer_layers", None)
    return _apply_deltanet_beta_decay(
        beta,
        mode=getattr(backbone, "deltanet_beta_decay", "none"),
        t0=int(getattr(backbone, "deltanet_beta_decay_t0", 1000)),
        position_dim=beta.ndim - 2,
    ).cpu()


def collect_effective_betas(model_name: str, model, train_x: torch.Tensor, train_y: torch.Tensor) -> list[dict]:
    records = []
    handles = []

    def hook_for(layer_index: int, layer_name: str):
        def hook(_module, _inputs, output):
            records.append(
                {
                    "layer_index": layer_index,
                    "layer": layer_name,
                    "beta": effective_beta(model, output),
                }
            )

        return hook

    for layer_index, (layer_name, layer) in enumerate(
        (item for item in model.named_modules() if isinstance(item[1], DeltaNet) and item[1].use_beta)
    ):
        handles.append(layer.b_proj.register_forward_hook(hook_for(layer_index, layer_name)))

    try:
        with torch.no_grad(), torch.autocast(
            device_type=device.type,
            dtype=autocast_dtype,
            enabled=use_autocast,
        ):
            _ = model.incontext_fit(train_x, train_y)
    finally:
        for handle in handles:
            handle.remove()

    print(f"{model_label(model_name)}: recorded {len(records)} DeltaNet beta tensors")
    return records


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
autocast_dtype = torch.bfloat16 if device.type == "cuda" and torch.cuda.is_bf16_supported() else torch.float16
use_autocast = device.type == "cuda"
model_configs = {model_name: find_model_config(model_name) for model_name in MODEL_NAMES}
print(f"device={device}, use_autocast={use_autocast}, autocast_dtype={autocast_dtype}")

base_model, base_config = load_model(MODEL_NAMES[0])
batch = base_config.priors[0].create_get_batch_method()(
    batch_size=BATCH_SIZE,
    seq_len=TRAIN_CONTEXT_LEN + N_TEST,
    num_features=NUM_FEATURES,
    single_eval_pos=TRAIN_CONTEXT_LEN,
    n_targets_per_input=1,
)
train_x = batch.x[:, :TRAIN_CONTEXT_LEN].to(device)
train_y = batch.y[:, :TRAIN_CONTEXT_LEN].to(device)
print(f"train_x={tuple(train_x.shape)}, train_y={tuple(train_y.shape)}")

beta_records_by_model = {}
for model_name in MODEL_NAMES:
    model = base_model if model_name == MODEL_NAMES[0] else load_model(model_name)[0]
    beta_records_by_model[model_name] = collect_effective_betas(model_name, model, train_x, train_y)
    if model_name != MODEL_NAMES[0]:
        del model
        if device.type == "cuda":
            torch.cuda.empty_cache()

for model_name, records in beta_records_by_model.items():
    if max(LAYER_INDICES) >= len(records):
        raise IndexError(f"{model_name} recorded only {len(records)} DeltaNet layers; requested {LAYER_INDICES}.")


## Plot Helpers

In [ ]:
MODEL_COLORS = {model_name: f"C{i}" for i, model_name in enumerate(MODEL_NAMES)}
BETA_FLOOR = torch.finfo(torch.float32).tiny


def mean_beta(record: dict) -> torch.Tensor:
    beta = record["beta"]
    seq_dim = beta.ndim - 2
    flat = beta.movedim(seq_dim, 0).reshape(beta.shape[seq_dim], -1)
    return flat.mean(dim=1).clamp_min(BETA_FLOOR)  # [seq_len]

def mean_cumulative_retention(record: dict, *, reverse: bool = False) -> torch.Tensor:
    beta = mean_beta(record).clamp(max=RETENTION_BETA_CEILING)
    log_retention = torch.log1p(-beta).double()
    if reverse:
        log_retention = log_retention.flip(0).cumsum(dim=0).flip(0) - log_retention
    else:
        log_retention = log_retention.cumsum(dim=0) - log_retention[0]
    return torch.exp(log_retention).clamp_min(RETENTION_FLOOR)


def sample_positions(n_positions: int, max_points: int, sampling: str | None = None) -> torch.Tensor:
    """Sample sequence positions linearly or logarithmically, returning at most max_points."""
    sampling = POSITION_SAMPLING if sampling is None else sampling
    if n_positions <= max_points:
        return torch.arange(n_positions)
    if sampling == "linear":
        return torch.linspace(0, n_positions - 1, max_points).round().long().unique()
    if sampling != "log":
        raise ValueError(f"Unknown position sampling mode {sampling!r}.")

    idx = torch.logspace(0, torch.log10(torch.tensor(float(n_positions))), max_points * 2)
    idx = (idx.round().long() - 1).clamp(0, n_positions - 1).unique()
    if idx.numel() > max_points:
        idx = idx[torch.linspace(0, idx.numel() - 1, max_points).round().long()]
    return idx


def smooth(values: torch.Tensor, window: int = SMOOTHING_WINDOW) -> torch.Tensor:
    """Optionally apply a moving average in log space with the given window size."""
    values = values.clamp_min(BETA_FLOOR)
    if not SMOOTH_BETA or window <= 1 or values.numel() < 3:
        return values
    window = min(int(window) | 1, values.numel() if values.numel() % 2 else values.numel() - 1)
    log_values = values.log10().view(1, 1, -1)
    kernel = torch.ones(1, 1, window, dtype=log_values.dtype) / float(window)
    return torch.pow(10.0, F.conv1d(F.pad(log_values, (window // 2, window // 2), mode="replicate"), kernel).flatten())


def position_edges(positions: torch.Tensor, scale: str) -> torch.Tensor:
    """Calculates the edges of positions based on the specified scale."""
    x = positions.float() + 1.0
    if scale == "log":
        x = x.log()
    elif scale != "linear":
        raise ValueError(f"Unknown axis scale {scale!r}.")

    if x.numel() == 1:
        edges = torch.tensor([float(x[0] - 0.5), float(x[0] + 0.5)])
    else:
        mid = (x[:-1] + x[1:]) / 2.0
        edges = torch.cat([(x[0] - (mid[0] - x[0]))[None], mid, (x[-1] + (x[-1] - mid[-1]))[None]])
    return edges.exp().clamp_min(1e-6) if scale == "log" else edges.clamp_min(0.5)



def retention_matrix(record: dict) -> tuple[torch.Tensor, torch.Tensor]:
    beta = mean_beta(record).clamp(max=RETENTION_BETA_CEILING)
    positions = sample_positions(beta.numel(), MAX_MATRIX_POINTS)
    log_cum = torch.log1p(-beta).cumsum(dim=0)[positions]
    matrix = torch.tril(torch.exp(log_cum[:, None] - log_cum[None, :]))
    return positions, matrix.clamp_min(RETENTION_FLOOR)


def combined_zoom_retention_matrix(record: dict) -> tuple[int, torch.Tensor, torch.Tensor]:
    beta = mean_beta(record).clamp(max=RETENTION_BETA_CEILING)
    n_positions = beta.numel()
    distances = sample_positions(n_positions, MAX_MATRIX_POINTS)
    positions = n_positions - 1 - distances
    log_cum = torch.log1p(-beta).cumsum(dim=0)

    target_grid = positions[:, None]
    source_grid = positions[None, :]
    valid = target_grid >= source_grid
    matrix = torch.exp(log_cum[target_grid] - log_cum[source_grid])
    matrix = torch.where(valid, matrix, torch.full_like(matrix, RETENTION_FLOOR))
    return n_positions, distances, matrix.clamp_min(RETENTION_FLOOR)


def retention_by_target_lag(record: dict) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    beta = mean_beta(record).clamp(max=RETENTION_BETA_CEILING)
    n_positions = beta.numel()
    target_positions = sample_positions(n_positions, MAX_MATRIX_POINTS)
    lags = sample_positions(n_positions, MAX_MATRIX_POINTS)
    log_cum = torch.log1p(-beta).cumsum(dim=0)

    source_positions = target_positions[:, None] - lags[None, :]
    valid = source_positions >= 0
    source_positions = source_positions.clamp_min(0).long()
    matrix = torch.exp(log_cum[target_positions, None] - log_cum[source_positions])
    matrix = torch.where(valid, matrix, torch.full_like(matrix, RETENTION_FLOOR))
    return target_positions, lags, matrix.clamp_min(RETENTION_FLOOR)


def final_state_retention_by_lag(record: dict) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    beta = mean_beta(record).clamp(max=RETENTION_BETA_CEILING)
    n_positions = beta.numel()
    target_distances = sample_positions(n_positions, MAX_MATRIX_POINTS)
    target_positions = n_positions - 1 - target_distances
    lags = sample_positions(n_positions, MAX_MATRIX_POINTS)
    log_cum = torch.log1p(-beta).cumsum(dim=0)

    source_positions = target_positions[:, None] - lags[None, :]
    valid = source_positions >= 0
    source_positions = source_positions.clamp_min(0).long()
    matrix = torch.exp(log_cum[target_positions, None] - log_cum[source_positions])
    matrix = torch.where(valid, matrix, torch.full_like(matrix, RETENTION_FLOOR))
    return target_distances, lags, matrix.clamp_min(RETENTION_FLOOR)


## Plots


In [ ]:
def beta_plot_label() -> str:
    return "smoothed mean effective beta" if SMOOTH_BETA else "mean effective beta"


def plot_beta_curves(layer_index: int) -> None:
    fig, ax = plt.subplots(figsize=(10, 3.5))
    ymax = 0.0
    for model_name in MODEL_NAMES:
        beta = mean_beta(beta_records_by_model[model_name][layer_index])
        idx = sample_positions(beta.numel(), MAX_CURVE_POINTS)
        y = smooth(beta[idx])
        ymax = max(ymax, float(y.max()))
        ax.plot(idx + 1, y, label=model_label(model_name), color=MODEL_COLORS[model_name], linewidth=1.8)

    ax.set_xscale("log")
    ax.set_ylim(0.0, ymax * 1.05)
    ax.set_title(f"{beta_plot_label()} | layer {layer_index}")
    ax.set_xlabel(f"sequence position ({POSITION_SAMPLING} sampled)")
    ax.set_ylabel("mean effective beta")
    ax.grid(True, which="both", alpha=0.25)
    ax.legend(fontsize=8)
    fig.tight_layout()
    plt.show()


def plot_beta_heatmap(model_name: str) -> None:
    records = beta_records_by_model[model_name]
    idx = sample_positions(mean_beta(records[0]).numel(), MAX_CURVE_POINTS)
    values = torch.stack([smooth(mean_beta(record)[idx]) for record in records])
    x_edges = position_edges(idx, "log")
    y_edges = torch.arange(len(records) + 1, dtype=torch.float32) - 0.5

    fig, ax = plt.subplots(figsize=(10, max(3.0, 0.33 * len(records))))
    image = ax.pcolormesh(
        x_edges.numpy(),
        y_edges.numpy(),
        values.numpy(),
        shading="auto",
        vmin=0.0,
        vmax=float(values.max() * 1.05),
    )
    ax.set_xscale("log")
    ax.set_yticks(range(len(records)))
    ax.set_yticklabels([f"Layer {record['layer_index']}" for record in records])
    ax.set_xlabel(f"sequence position (log scale, {POSITION_SAMPLING} sampled)")
    ax.set_ylabel("DeltaNet layer")
    ax.set_title(f"{model_name} | {beta_plot_label()} by layer")
    fig.colorbar(image, ax=ax, label="mean effective beta")
    fig.tight_layout()
    plt.show()


def plot_cumulative_retention_heatmap(model_name: str, *, reverse: bool = False) -> None:
    records = beta_records_by_model[model_name]
    values = torch.stack([mean_cumulative_retention(record, reverse=reverse) for record in records])
    n_positions = values.shape[1]

    if reverse:
        x_positions = sample_positions(n_positions, MAX_CURVE_POINTS)
        idx = n_positions - 1 - x_positions
        values = values[:, idx]
        xlabel = f"positions before final token (log scale, {POSITION_SAMPLING} sampled; final=1)"
    else:
        idx = sample_positions(n_positions, MAX_CURVE_POINTS)
        x_positions = idx
        values = values[:, idx]
        xlabel = f"sequence position (log scale, {POSITION_SAMPLING} sampled)"

    x_edges = position_edges(x_positions, "log")
    y_edges = torch.arange(len(records) + 1, dtype=torch.float32) - 0.5

    fig, ax = plt.subplots(figsize=(10, max(3.0, 0.33 * len(records))))
    image = ax.pcolormesh(
        x_edges.numpy(),
        y_edges.numpy(),
        values.numpy(),
        shading="auto",
        norm=LogNorm(vmin=RETENTION_FLOOR, vmax=1.0),
    )
    ax.set_xscale("log")
    ax.set_xlim(1.0, float(x_edges[-1]))
    ax.set_yticks(range(len(records)))
    ax.set_yticklabels([f"Layer {record['layer_index']}" for record in records])
    ax.set_xlabel(xlabel)
    ax.set_ylabel("DeltaNet layer")
    direction = "reverse cumulative retention" if reverse else "cumulative retention"
    ax.set_title(f"{model_name} | {direction} by layer")
    fig.colorbar(image, ax=ax, label="mean cumprod 1 - beta")
    fig.tight_layout()
    plt.show()


def plot_retention_by_lag(model_name: str, layer_index: int) -> None:
    target_positions, lags, matrix = retention_by_target_lag(beta_records_by_model[model_name][layer_index])
    x_edges = position_edges(target_positions, "log")
    y_edges = position_edges(lags, "log")

    fig, ax = plt.subplots(figsize=(7.0, 5.4))
    image = ax.pcolormesh(
        x_edges.numpy(),
        y_edges.numpy(),
        matrix.T.numpy(),
        shading="auto",
        norm=LogNorm(vmin=RETENTION_FLOOR, vmax=1.0),
    )
    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_xlim(1.0, float(x_edges[-1]))
    ax.set_ylim(1.0, float(y_edges[-1]))
    ax.set_xlabel("target sequence position")
    ax.set_ylabel("lag = target - source")
    ax.set_title(f"{model_label(model_name)} | layer {layer_index} retention horizon by target")
    fig.colorbar(image, ax=ax, label="retention from source to target")
    fig.tight_layout()
    plt.show()


def plot_final_state_retention_by_lag(model_name: str, layer_index: int) -> None:
    target_distances, lags, matrix = final_state_retention_by_lag(beta_records_by_model[model_name][layer_index])
    x_edges = position_edges(target_distances, "log")
    y_edges = position_edges(lags, "log")

    fig, ax = plt.subplots(figsize=(7.0, 5.4))
    image = ax.pcolormesh(
        x_edges.numpy(),
        y_edges.numpy(),
        matrix.T.numpy(),
        shading="auto",
        norm=LogNorm(vmin=RETENTION_FLOOR, vmax=1.0),
    )
    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_xlim(1.0, float(x_edges[-1]))
    ax.set_ylim(1.0, float(y_edges[-1]))
    ax.set_xlabel(f"target distance before final token ({POSITION_SAMPLING} sampled; final=1)")
    ax.set_ylabel("lag = target - source")
    ax.set_title(f"{model_label(model_name)} | layer {layer_index} final-state retention horizon")
    fig.colorbar(image, ax=ax, label="retention from source to target")
    fig.tight_layout()
    plt.show()


def plot_retention_matrix(model_name: str, layer_index: int, scale: str, *, reverse: bool = False) -> None:
    record = beta_records_by_model[model_name][layer_index]
    if reverse:
        n_positions, distances, matrix = combined_zoom_retention_matrix(record)
        x_edges = position_edges(distances, scale)
        y_edges = x_edges
    else:
        positions, matrix = retention_matrix(record)
        x_edges = position_edges(positions, scale)
        y_edges = x_edges

    fig, ax = plt.subplots(figsize=(6.5, 5.7))
    image = ax.pcolormesh(
        x_edges.numpy(),
        y_edges.numpy(),
        matrix.numpy(),
        shading="auto",
        norm=LogNorm(vmin=RETENTION_FLOOR, vmax=1.0),
    )
    if scale == "log":
        ax.set_xscale("log")
        ax.set_yscale("log")
        if reverse:
            ax.set_xlim(float(x_edges[-1]), 1.0)
            ax.set_ylim(1.0, float(y_edges[-1]))
            ticks = torch.tensor([1, 10, 100, 1_000, 10_000, 100_000, n_positions], dtype=torch.float32)
            ticks = ticks[(ticks >= 1) & (ticks <= n_positions)].unique()
            tick_labels = [f"{int(n_positions - tick.item() + 1):,}" for tick in ticks]
            ax.set_xticks(ticks.tolist())
            ax.set_yticks(ticks.tolist())
            ax.set_xticklabels(tick_labels, rotation=30, ha="right")
            ax.set_yticklabels(tick_labels)
        else:
            ax.set_xlim(1.0, float(x_edges[-1]))
            ax.set_ylim(float(y_edges[-1]), 1.0)
    else:
        ax.invert_yaxis()

    ax.set_xlabel("source sequence position" + (" (reverse-log zoom near end)" if reverse else ""))
    ax.set_ylabel("target sequence position" + (" (reverse-log zoom near end)" if reverse else ""))
    direction = "retention matrix, reverse-log end zoom" if reverse else "retention matrix"
    ax.set_title(f"{model_label(model_name)} | layer {layer_index} {direction} ({scale}-{scale})")
    fig.colorbar(image, ax=ax, label="prod(1 - beta)")
    fig.tight_layout()
    plt.show()


## Render


In [ ]:
for layer_index in LAYER_INDICES:
    plot_beta_curves(layer_index)

for model_name in MODEL_NAMES:
    plot_beta_heatmap(model_name)

for model_name in MODEL_NAMES:
    plot_cumulative_retention_heatmap(model_name, reverse=False)
    plot_cumulative_retention_heatmap(model_name, reverse=True)

for model_name in MODEL_NAMES:
    for layer_index in LAYER_INDICES:
        for scale in MATRIX_AXIS_SCALES:
            plot_retention_matrix(model_name, layer_index, scale, reverse=False)
            plot_retention_matrix(model_name, layer_index, scale, reverse=True)

for model_name in MODEL_NAMES:
    for layer_index in LAYER_INDICES:
        plot_retention_by_lag(model_name, layer_index)

for model_name in MODEL_NAMES:
    for layer_index in LAYER_INDICES:
        plot_final_state_retention_by_lag(model_name, layer_index)


In [ ]:
import numpy as np
from matplotlib.ticker import FuncFormatter, MaxNLocator


def format_sequence_tick(value, _position=None):
    value = int(round(float(value)))
    if abs(value) >= 1000:
        return f"{value / 1000:g}k"
    return str(value)


def plot_banded(M, scale=1.0, log=False, ax=None, num_ticks=10, axis_unit=1.0, **kwargs):
    M = np.asarray(M, dtype=float)
    if M.ndim != 2 or M.shape[0] != M.shape[1]:
        raise ValueError("M must be a square 2D matrix.")
    if log and scale <= 0:
        raise ValueError("scale must be positive when log=True.")
    if axis_unit <= 0:
        raise ValueError("axis_unit must be positive.")

    N = M.shape[0]
    to_y = lambda K: N * np.log1p(K * scale / N) / np.log1p(scale) if log else K * scale
    to_k = lambda y: N * np.expm1(y * np.log1p(scale) / N) / scale if log else y / scale

    arange = np.arange(N + 1, dtype=float)
    I, J = np.meshgrid(arange, arange, indexing="ij")

    K = np.abs(I - J)
    Z = np.maximum(N - K, 1.0)
    K = np.where(I > J, to_y(K), K)
    base = np.minimum(I, J) * (N - K) / Z
    I, J = base + K * (I > J), base + K * (I < J)

    if ax is None:
        _, ax = plt.subplots()
    kwargs.setdefault("shading", "flat")
    kwargs.setdefault("rasterized", True)
    kwargs.setdefault("cmap", "viridis")
    mesh = ax.pcolormesh(J * axis_unit, I * axis_unit, M, **kwargs)

    ax.set_aspect("equal")
    ax.set_xlim(0, N * axis_unit)
    ax.set_ylim(N * axis_unit, 0)

    ax.set_xlabel("source position", labelpad=8)
    ax.xaxis.set_ticks_position("top")
    ax.xaxis.set_label_position("top")
    ax.tick_params(axis="x", which="major", pad=6)
    ax.spines["top"].set_visible(True)

    ax.set_ylabel("target position", labelpad=10, rotation=270, va="bottom")
    ax.yaxis.set_ticks_position("right")
    ax.yaxis.set_label_position("right")
    ax.tick_params(axis="y", which="major", pad=6)
    ax.spines["right"].set_visible(True)

    ax_l = ax.secondary_yaxis(
        "left",
        functions=(lambda y: to_k(y / axis_unit) * axis_unit, lambda K: to_y(K / axis_unit) * axis_unit),
    )
    ax_l.set_ylabel("lag (target - source)", labelpad=10)
    ax_l.tick_params(axis="y", which="major", pad=5)
    ax_b = ax.secondary_xaxis(
        "bottom",
        functions=(
            lambda x: to_k(N - x / axis_unit) * axis_unit,
            lambda K: (N - to_y(K / axis_unit)) * axis_unit,
        ),
    )
    ax_b.set_xlabel("lag (target - source)", labelpad=8)
    ax_b.tick_params(axis="x", which="major", pad=6)

    if log:
        k_max = N * axis_unit
        hi = int(np.log10(k_max))
        major = 10.0 ** np.arange(hi + 1)
        major = major[major <= k_max]
        major = major[np.append(np.diff(to_y(major / axis_unit) / N) >= 0.05, True)]
        minor = np.array([k * 10.0**e for e in range(-1, hi + 1) for k in range(2, 10)])
        minor = minor[(minor > major[0]) & (minor < major[-1])]
        for axis in (ax_l.yaxis, ax_b.xaxis):
            axis.set_ticks(major, [f"$10^{{{int(np.log10(k))}}}$" for k in major])
            axis.set_ticks(minor, minor=True)

    ax.xaxis.set_major_locator(MaxNLocator(nbins=num_ticks, integer=True))
    ax.yaxis.set_major_locator(MaxNLocator(nbins=num_ticks, integer=True))
    ax.xaxis.set_major_formatter(FuncFormatter(format_sequence_tick))
    ax.yaxis.set_major_formatter(FuncFormatter(format_sequence_tick))
    grid_kwargs = dict(color=plt.rcParams["grid.color"], lw=plt.rcParams["grid.linewidth"])
    for tick in ax.get_xticks():
        idx = int(round(tick / axis_unit))
        if 0 <= idx <= N:
            ax.plot(J[:, idx] * axis_unit, I[:, idx] * axis_unit, **grid_kwargs)
            ax.plot(J[N - idx, :] * axis_unit, I[N - idx, :] * axis_unit, **grid_kwargs)
    return mesh


def banded_retention_matrix(record: dict, max_points: int = MAX_MATRIX_POINTS) -> tuple[torch.Tensor, torch.Tensor, int]:
    beta = mean_beta(record).clamp(max=RETENTION_BETA_CEILING)
    positions = sample_positions(beta.numel(), max_points, sampling="linear")
    log_cum = torch.log1p(-beta).cumsum(dim=0)[positions]
    matrix = torch.tril(torch.exp(log_cum[:, None] - log_cum[None, :]))
    return positions, matrix.clamp_min(RETENTION_FLOOR), beta.numel()


def plot_banded_retention_matrix(
    model_name: str,
    layer_index: int,
    *,
    scale: float = 1000.0,
    log: bool = True,
    max_points: int = MAX_MATRIX_POINTS,
) -> None:
    positions, matrix, n_positions = banded_retention_matrix(
        beta_records_by_model[model_name][layer_index], max_points=max_points
    )
    axis_unit = n_positions / positions.numel()

    fig, ax = plt.subplots(figsize=(10.5, 8.5))
    mesh = plot_banded(
        matrix.numpy(),
        scale=scale,
        log=log,
        ax=ax,
        num_ticks=8,
        norm=LogNorm(vmin=RETENTION_FLOOR, vmax=1.0),
        axis_unit=axis_unit,
    )
    ax.set_title(
        f"{model_label(model_name)} | layer {layer_index} source-to-target retention"
        f" ({positions.numel():,} samples across {n_positions:,} steps)",
        pad=16,
    )
    fig.colorbar(
        mesh,
        ax=ax,
        label="contribution of source to target",
        fraction=0.04,
        pad=0.12,
    )
    fig.subplots_adjust(left=0.15, right=0.78, top=0.86, bottom=0.16)
    plt.show()


for model_name in MODEL_NAMES:
    for layer_index in LAYER_INDICES:
        plot_banded_retention_matrix(model_name, layer_index)
